# 12b — Pitch calibration at imgsz 1280

Identical to notebook 12 except `IMGSZ = 1280`. Phase 1 found training resolution to be the largest lever for small objects; notebook 14 showed the remaining calibration error is keypoint localisation (the confidence threshold is not the lever: 0.5 is best on val). Batch 16 may not fit the T4 at 1280 — the batch actually used is recorded.

# 12 — Automatic pitch calibration (Phase 4)

A YOLO11s-pose model finds the 32 pitch keypoints of `ballhawk_pitch.KEYPOINTS` in each frame; `bp.frame_homography` fits an image→pitch homography with RANSAC. Frames with fewer than 4 confident keypoints are marked uncalibrated, never guessed.

**Data:** HF `martinjolif/football-pitch-detection` (CC BY 4.0), re-split by video (`splits_pitch.json`). The mirror's own split shares videos between train and test, and two of the three Phase 3 click frames are *in* that dataset — so both click-frame videos (121364, 744b27) are forced into **test**: the model never sees them. 213 / 54 / 50 images.

**Keypoint order verified** visually against `bp.KEYPOINTS` before training. **Real-world coordinates** use the official Laws of the Game dimensions and an assumed 105 × 68 m pitch (points on the outer lines and halfway line depend on pitch size; penalty-area points do not).

**Validation (test split, never trained on):**
- pose mAP (ultralytics), and per-keypoint pixel error of confident predictions vs labels;
- calibration success rate per frame;
- **reprojection error in metres:** each labelled keypoint is projected through the *predicted* homography and compared with its official pitch position — the labels never enter the fit;
- later, against Ro's manual clicks (notebook 10) on the same frames.

**Video:** success rate and temporal jitter on the notebook 08 clip, raw vs exponentially smoothed homography. That clip is handheld phone footage from a low stand angle — a domain shift, stated as such.

In [ ]:
SMOKE = False  # True: 1 epoch on 25% of train
IMGSZ = 1280
EPOCHS = 150
KP_CONF = 0.5
VIDEO = "/content/video/france_2018.mp4"  # built by notebook 08
EMA = 0.3  # weight of the newest frame when smoothing homographies
REUSE_WEIGHTS = True  # evaluate an existing best.pt instead of retraining (pitch coordinates changed, not the model)

In [ ]:
import sys, json, time, pathlib
sys.path.insert(0, "/content/ballhawk")
import numpy as np, pandas as pd, yaml, cv2
from ultralytics import YOLO
import ballhawk_common as bc
import ballhawk_pitch as bp
import ballhawk_video as bv

data_yaml = bc.prepare_data(bc.PITCH_DATA_DIR, "splits_pitch.json", bc.PITCH_RAW_DIR)
out = bc.RESULTS_DIR / ("pitch_kp_yolo11s" + ("" if IMGSZ == 960 else f"_{IMGSZ}") + ("_smoke" if SMOKE else "")); out.mkdir(parents=True, exist_ok=True)

In [ ]:
args = dict(data=data_yaml, imgsz=IMGSZ, epochs=1 if SMOKE else EPOCHS, batch=16, seed=0, deterministic=True,
            project="/content/runs", name=out.name, exist_ok=True)
if SMOKE:
    args["fraction"] = 0.25
if REUSE_WEIGHTS and (out / "best.pt").exists() and (out / "pitch_metrics.json").exists():
    prev = json.loads((out / "pitch_metrics.json").read_text())
    best, train_min, batch_used = out / "best.pt", prev["train_minutes"], prev["batch"]
    print(f"reusing {best} (trained {train_min:.1f} min, batch {batch_used})")
else:
    model = YOLO("yolo11s-pose.pt")
    t0 = time.time(); model.train(**args); train_min = (time.time() - t0) / 60
    save_dir = pathlib.Path(model.trainer.save_dir)  # Pitfall 2
    best = save_dir / "weights" / "best.pt"
    batch_used = model.trainer.batch_size
    for f in ["weights/best.pt", "results.csv", "results.png", "args.yaml"]:
        if (save_dir / f).exists():
            (out / pathlib.Path(f).name).write_bytes((save_dir / f).read_bytes())
m = YOLO(best).val(data=data_yaml, split="test", imgsz=IMGSZ, plots=False, verbose=False)
pose = {"pose_mAP50": float(m.pose.map50), "pose_mAP50_95": float(m.pose.map), "box_mAP50": float(m.box.map50)}
print(f"batch used {batch_used}, {train_min:.1f} min;", pose)

In [ ]:
# Test split: keypoint pixel error, calibration success, reprojection error in metres.
kp_model = YOLO(best)
cfg = yaml.safe_load(open(data_yaml))
rows = []
for img in bc.split_images(data_yaml, "test"):
    im = cv2.imread(str(img)); h, w = im.shape[:2]
    lab = np.array(open(img.parents[1] / "labels" / f"{img.stem}.txt").read().split(), float)[5:].reshape(-1, 3)
    gt_vis = np.flatnonzero(lab[:, 2] > 0); gt_xy = lab[:, :2] * [w, h]
    idx, xy = bp.detect_keypoints(kp_model, im, IMGSZ, KP_CONF)
    common = np.intersect1d(idx, gt_vis)
    px_err = [float(np.linalg.norm(xy[list(idx).index(i)] - gt_xy[i])) for i in common]
    fit = bp.frame_homography(kp_model, im, IMGSZ, KP_CONF)
    metres = bp.reprojection_error(fit[0], gt_xy[gt_vis], bp.PITCH_XY[gt_vis])[0].tolist() if fit else []
    rows.append({"image": img.name, "kp_found": len(idx), "gt_visible": len(gt_vis), "calibrated": fit is not None,
                 "median_px_err": float(np.median(px_err)) if px_err else np.nan,
                 "median_reproj_m": float(np.median(metres)) if metres else np.nan})
test_df = pd.DataFrame(rows)
summary = {"calibration_success": float(test_df.calibrated.mean()),
           "median_kp_px_err": float(test_df.median_px_err.median()),
           "median_reproj_m": float(test_df.median_reproj_m.median()),
           "p90_reproj_m": float(test_df.median_reproj_m.quantile(0.9))}
test_df.to_csv(out / "test_frames.csv", index=False)
print(summary)

In [ ]:
# Video: per-frame calibration success and temporal jitter, raw vs EMA-smoothed homography.
# Jitter = median |second difference| of where a fixed image probe lands on the pitch (metres/frame^2):
# smooth camera motion gives a small second difference, frame-to-frame fitting noise does not.
# Probe = median image position of all RANSAC-inlier keypoints over the clip: on the grass by
# construction (the image centre can sit above the far touchline, where projections explode).
cap = cv2.VideoCapture(bv.fetch_clip(VIDEO)); Hs, inlier_xy = [], []
while True:
    got, frame = cap.read()
    if not got:
        break
    fit = bp.frame_homography(kp_model, frame, IMGSZ, KP_CONF)
    Hs.append(None if fit is None else fit[0] / fit[0][2, 2])
    if fit is not None:
        inlier_xy.append(fit[2][fit[3]])
probe = [np.median(np.vstack(inlier_xy), axis=0)] if inlier_xy else [[0.0, 0.0]]
raw, smooth, H_s = [], [], None
for H in Hs:
    if H is None:
        raw.append(None); smooth.append(None); continue
    H_s = H if H_s is None else EMA * H + (1 - EMA) * H_s
    raw.append(bp.project(H, probe)[0]); smooth.append(bp.project(H_s, probe)[0])
def jitter(track):
    d2 = [np.linalg.norm(track[i + 1] - 2 * track[i] + track[i - 1]) for i in range(1, len(track) - 1)
          if all(track[j] is not None for j in (i - 1, i, i + 1))]
    return float(np.median(d2)) if d2 else float("nan")
video = {"frames": len(Hs), "calibration_success": float(np.mean([H is not None for H in Hs])) if Hs else 0.0,
         "probe_image_xy": [float(v) for v in probe[0]], "jitter_raw_m": jitter(raw), "jitter_ema_m": jitter(smooth),
         "ema_weight": EMA}
print(video)

In [ ]:
record = {"model": "yolo11s-pose.pt", "imgsz": IMGSZ, "epochs": args["epochs"], "batch": batch_used,
          "train_minutes": train_min, "kp_conf": KP_CONF, "split": "test", **pose, **summary,
          "video_clip": VIDEO, "video": video,
          "notes": ("Pitch split by video (splits_pitch.json), Phase 3 click-frame videos forced into test. "
                    "Real-world coordinates: official dimensions, 105 x 68 m assumed; keypoints 10/11/18/19 are penalty-arc x box-front "
                    "intersections (roboflow/sports mislabels them as goal-area y). ultralytics 8.4.163.")}
(out / "pitch_metrics.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))